# Week 98, Day 3: System design

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w098d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
keys = range(10_000)
before = {k: k % 4 for k in keys}
after = {k: k % 5 for k in keys}
print(sum(before[k] != after[k] for k in keys) / 10_000)

> **Key idea.** Consistent hashing with virtual nodes moves about 1/(n+1) of keys when a server is added, not most of them. R + W > N gives reads that see the latest write; N=3, R=2, W=2 survives one failure. Little's law: L = λW; arrivals above capacity grow a queue without limit. Series availabilities multiply; parallel ones fail only together. Fan-out on write suits ordinary accounts, on read the very popular.

### Exercise: A ring that moves little

Write `ring_owner(key, servers, vnodes=100)`: hash each server's virtual nodes `f"{server}#{i}"` and the key with `int(hashlib.md5(text.encode()).hexdigest(), 16)`, and return the server owning the first point at or after the key's hash on the ring (wrapping round). Then `moved_fraction(keys, before, after, vnodes=100)`: the fraction of keys whose owner changes.

In [ ]:
import bisect
import hashlib

def ring_owner(key, servers, vnodes=100):
    ___

def moved_fraction(keys, before, after, vnodes=100):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ring():\n    \"stable and spread\"\n    servers = [\"a\", \"b\", \"c\", \"d\"]\n    owners = [ring_owner(f\"walk-{i}\", servers) for i in range(4000)]\n    assert owners == [ring_owner(f\"walk-{i}\", servers) for i in range(4000)]\n    counts = [owners.count(s) for s in servers]\n    assert min(counts) > 600, counts\n\ndef test_moves():\n    \"adding a fifth server\"\n    keys = [f\"walk-{i}\" for i in range(3000)]\n    moved = moved_fraction(keys, [\"a\", \"b\", \"c\", \"d\"], [\"a\", \"b\", \"c\", \"d\", \"e\"])\n    assert 0.1 < moved < 0.3, moved\n    assert moved_fraction(keys, [\"a\", \"b\"], [\"a\", \"b\"]) == 0")

### Exercise: Quorums

Write `quorum(n, r, w)`: a dict with `consistent` (whether `r + w > n`), `read_failures` and `write_failures` (how many copies can be down while reads, or writes, still succeed: `n - r` and `n - w`). Raise `ValueError` unless `1 <= r <= n` and `1 <= w <= n`.

In [ ]:
def quorum(n, r, w):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_quorum():\n    \"R + W > N\"\n    assert quorum(3, 2, 2) == {\"consistent\": True, \"read_failures\": 1, \"write_failures\": 1}\n    assert quorum(3, 1, 1) == {\"consistent\": False, \"read_failures\": 2, \"write_failures\": 2}\n    assert quorum(5, 3, 3)[\"consistent\"] and not quorum(4, 2, 2)[\"consistent\"]\n    for bad in [(3, 0, 2), (3, 4, 1)]:\n        try:\n            quorum(*bad)\n        except ValueError:\n            pass\n        else:\n            raise AssertionError(bad)")

### Exercise: Little's law and a growing queue

Write `in_progress(rate_per_s, seconds_each)`: Little's law, rounded to 1 decimal place. Then `backlog(arrivals_per_s, capacity_per_s, seconds, start=0)`: the queue length after `seconds`, growing by the excess each second and never below 0.

In [ ]:
def in_progress(rate_per_s, seconds_each):
    ___

def backlog(arrivals_per_s, capacity_per_s, seconds, start=0):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_queues():\n    \"L = lambda W\"\n    assert in_progress(200, 0.05) == 10.0 and in_progress(1500, 0.12) == 180.0\n    assert backlog(120, 100, 60) == 1200 and backlog(80, 100, 60, start=500) == 0 and backlog(80, 100, 10, start=500) == 300")

### Exercise: Nines, in series and in parallel

Write `downtime_minutes(availability)`: the minutes a year (365 days) outside the given availability (a fraction such as `0.999`), rounded to 1 decimal place. Then `series(*parts)` (all must work: multiply) and `parallel(*parts)` (any one is enough: one minus the product of the failure chances), both rounded to 6 decimal places.

In [ ]:
def downtime_minutes(availability):
    ___

def series(*parts):
    ___

def parallel(*parts):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_nines():\n    \"multiply in series\"\n    assert downtime_minutes(0.999) == 525.6 and downtime_minutes(0.9999) == 52.6 and downtime_minutes(1) == 0\n    assert series(0.999, 0.999) == 0.998001 and series(0.99, 0.999, 0.9999) == 0.988911\n    assert parallel(0.99, 0.99) == 0.9999 and parallel(0.9) == 0.9")

### Exercise: Fan out on write or on read?

Write `fanout_costs(posts_per_day, followers_each, reads_per_day, follows_each)`: return `(write_cost, read_cost, choice)`: feed rows written a day with fan-out on write (`posts × followers`), rows read a day with fan-out on read (`reads × follows`), and `"write"` or `"read"` for the cheaper (ties `"write"`).

In [ ]:
def fanout_costs(posts_per_day, followers_each, reads_per_day, follows_each):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fanout():\n    \"ordinary accounts and celebrities\"\n    assert fanout_costs(1_000_000, 200, 50_000_000, 150) == (200_000_000, 7_500_000_000, \"write\")\n    assert fanout_costs(100, 30_000_000, 50_000_000, 1) == (3_000_000_000, 50_000_000, \"read\")\n    assert fanout_costs(1, 2, 1, 2)[2] == \"write\"")

### Bug hunt: A quorum that only touched

With N = 4, R = 2 and W = 2 a read and a write can use different copies, so reads may be stale; `>=` calls it consistent and prints `True`. Fix it so it prints `False`.

In [ ]:
n, r, w = 4, 2, 2
print(r + w >= n)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"overlap needs more than N\"\n    expect_output(\"False\")")

### Bug hunt: Availability added up

Two services in a chain don't add their availabilities; this prints `1.998`, which isn't even a probability. Multiply them so it prints `0.998001`.

In [ ]:
a, b = 0.999, 0.999
print(round(a + b, 6))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"series multiplies\"\n    expect_output(\"0.998001\")")

### Puzzle: Finding the owner on the ring

Arrange the lookup. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # quorum(3, 2, 2)["consistent"]
q = ___   # in_progress(100, 0.2)
r = ___   # downtime_minutes(0.99) rounded to the nearest whole minute (an int)
s = ___   # True or False: hash % servers moves few keys when a server is added
t = ___   # the fan-out choice for an account with 30 million followers ("write" or "read")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q == 20.0 and r == 5256, \"p, q and r\"\n    assert s is False and t == \"read\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: sizing a replicated key-value store

Write `kv_plan(req)` using today's helpers (given). `req` has `items`, `bytes_each`, `copies` (N), `node_tb` (usable terabytes per node), `node_availability`, `reads_per_s`, `ms_each` (time per read) and `target_minutes` (allowed downtime a year). Return lines:

1. `"data: D TB raw, T TB with N copies"` (TB of 10^12 bytes, 1 decimal place);
2. `"nodes: K"`: enough for the copies at 70% full (`ceil(T / (node_tb × 0.7))`), and at least N;
3. `"quorum: R=X W=Y (consistent, survives F failure(s))"`: the smallest equal R and W with R + W > N, F = N − R;
4. `"reads in progress: L"` (Little's law);
5. `"item availability: A"`: the chance at least R of the N copies are up, from `node_availability` (sum the binomial probabilities), to 6 decimal places, and `"downtime: M min a year (target T)"`, M from `downtime_minutes`;
6. `"verdict: meets target"` or `"verdict: misses target"`.

In [ ]:
import math

def downtime_minutes(availability):
    return round((1 - availability) * 365 * 24 * 60, 1)

def in_progress(rate_per_s, seconds_each):
    return round(rate_per_s * seconds_each, 1)

def kv_plan(req):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("REQ = {\"items\": 2_000_000_000, \"bytes_each\": 1_000, \"copies\": 3, \"node_tb\": 4, \"node_availability\": 0.99,\n       \"reads_per_s\": 50_000, \"ms_each\": 4, \"target_minutes\": 60}\n\ndef test_plan():\n    \"three copies, two must agree\"\n    assert kv_plan(REQ) == [\n        \"data: 2.0 TB raw, 6.0 TB with 3 copies\",\n        \"nodes: 3\",\n        \"quorum: R=2 W=2 (consistent, survives 1 failure)\",\n        \"reads in progress: 200.0\",\n        \"item availability: 0.999702\",\n        \"downtime: 156.6 min a year (target 60)\",\n        \"verdict: misses target\",\n    ]\n\ndef test_five():\n    \"five copies\"\n    lines = kv_plan({**REQ, \"copies\": 5, \"items\": 20_000_000_000})\n    assert lines[1] == \"nodes: 36\" and lines[2] == \"quorum: R=3 W=3 (consistent, survives 2 failures)\" and lines[-1] == \"verdict: meets target\"")

## Warm-up

Spreading keys with `hash % servers` is simple until a server is added: 80% of the keys move to a different machine, and the caches and databases behind them have to move too. **Consistent hashing** moves only about a fifth when going from four servers to five.

## Spreading data

Consistent hashing (from David Karger and colleagues, 1997) places servers at points on a **ring** of hash values; each key belongs to the first server clockwise from its own hash. Adding a server takes over only the keys between it and its neighbour. With few servers the ring is lumpy, so each server is placed at many points (**virtual nodes**), which evens out the load. Week 81 built one; today you'll measure it.

## Copies that agree

With **N** copies of each item, a write waits for **W** copies to confirm and a read asks **R** copies. If **R + W > N**, every read overlaps every write in at least one copy, so a read always sees the latest write (when copies carry versions). N = 3, W = 2, R = 2 is the common choice: it survives one copy failing for reads and for writes. W = 1 and R = 1 is fast and available, but reads may be stale: **eventual consistency**.

## Queues and downtime

**Little's law** (John Little, 1961): the average number of items in a system is the arrival rate times the average time each spends there, L = λW. A service handling 200 requests a second at 50 ms each has about 10 in progress; if arrivals exceed the service rate, the queue grows without limit, by the difference every second.

**Availability**: 99.9% ("three nines") allows 8.8 hours of downtime a year; 99.99% allows 53 minutes. Parts in **series** (all must work) multiply availabilities, so two 99.9% services in a chain give 99.8%. Parts in **parallel** (any one is enough) fail only together: 1 − (1 − a)².

**Fan-out**: a feed can be built when someone posts (**fan-out on write**: copy the post into every follower's feed, cheap reads, expensive for accounts with millions of followers) or when someone reads (**fan-out on read**: gather from everyone they follow, cheap writes, expensive reads). Large systems mix them: write for ordinary accounts, read for very popular ones.

## Practice

## Quiz

## Daily challenge

## A mock round

Forty-five minutes, aloud, on **"Design the storage behind Waymark's saved routes for ten million users"**: requirements, estimates (use `kv_plan`), API, data model, the ring and the quorum, and the trade-offs of R and W. Then **"Design a news feed"**, deciding fan-out on write or read with `fanout_costs`.

## Before you go

Tomorrow is a debugging interview: a failing program, a traceback, and twenty minutes to find out why.